In [1]:
from xml.parsers.expat import model
import yaml
import pandas as pd
import numpy as np
import os
import sys
from datetime import datetime
current_dir = os.path.dirname(os.path.abspath('.'))
project_root = os.path.abspath(os.path.join(current_dir, "../.."))
sys.path.insert(0, project_root)

from functions.make_dataset import save_data
from utils.utils import to_jsonl
from functions.train_model import save_model
from functions.evaluate_model import evaluate_reg_model, MetricsOrchestrator
from functions.predict_model import make_prediction_reg
from functions.ann_model import KerasRegressor

c:\Users\gustavo\anaconda3\envs\mf_tf\lib\site-packages\dask\dataframe\_pyarrow_compat.py:23: UserWarning: You are using pyarrow version 11.0.0 which is known to be insecure. See https://www.cve.org/CVERecord?id=CVE-2023-47248 for further details. Please upgrade to pyarrow>=14.0.1 or install pyarrow-hotfix to patch your current version.
  warnings.warn(


In [2]:
# 1. Carregar configurações
with open(os.path.join(project_root, "Regression/house_prices/config/config.yaml"), "r") as f:
    config = yaml.safe_load(f)

# pipeline selection    
with open(os.path.join(project_root, "Regression/house_prices/config/pipeline.yaml"), "r") as f:
    config_pipe = yaml.safe_load(f)

# model selection    
with open(os.path.join(project_root, "Regression/house_prices/config/model.yaml"), "r") as f:
    config_model = yaml.safe_load(f)

In [3]:
pipeline_name='pipeline1'

In [4]:
X_train = pd.read_parquet(
    os.path.join(
        config['init_path'],
        config['data']['feature_eng'],
        f"X_train_feat_eng_{pipeline_name}.parquet")
)

y_train = pd.read_parquet(
    os.path.join(
        config['init_path'],
        config['data']['feature_eng'],
        f"y_train_feat_eng_{pipeline_name}.parquet")
)

X_val = pd.read_parquet(
    os.path.join(
        config['init_path'],
        config['data']['feature_eng'],
        f"X_val_feat_eng_{pipeline_name}.parquet")
)

y_val = pd.read_parquet(
    os.path.join(
        config['init_path'],
        config['data']['feature_eng'],
        f"y_val_feat_eng_{pipeline_name}.parquet")
)

In [5]:
y_train

,SalePrice
254,11.884496
1066,12.089544
638,11.350418
799,12.072547
380,11.751950
...,...
1095,12.080696
1130,11.813037
1294,11.652696
860,12.154521


In [5]:
model_name='ann'

In [6]:
model = KerasRegressor(
    input_dim=X_train.shape[1], 
    hidden_units=(32, 16),
    nonlinear=True,
    epochs=100)    

    
model_info = [{
    'model':model_name,
    'best_paramns': model.get_params(),
    'undersamplig': None,
    'model_type':model_name,
    'timestamp': datetime.now().isoformat()        
}]       

In [7]:
print(model.get_params(), end='\n')

{'batch_size': 32, 'dropout_rate': 0.1, 'epochs': 100, 'hidden_units': (32, 16), 'input_dim': 16, 'learning_rate': 0.001, 'nonlinear': True, 'verbose': 0}


In [8]:
model_reg = model.fit(X_train, y_train)

In [9]:
             
    
# 5. Evaulate model
metrics_train = evaluate_reg_model(model_reg, X_train, y_train)

In [10]:
metrics_train

{'mean_absolute_error': 0.13483498320728643,
 'mean_squared_error': 0.032637162737544446,
 'root_mean_squared_error': 0.1806575842237033,
 'r2_score': 0.7859050035476685}

In [11]:
metrics_val = evaluate_reg_model(model_reg, X_val, y_val)

In [12]:
metrics_val

{'mean_absolute_error': 0.1545053125250127,
 'mean_squared_error': 0.04894480920784257,
 'root_mean_squared_error': 0.2212347377964016,
 'r2_score': 0.7377175092697144}